[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/14_decision_probe.ipynb)

# 14 · Decision 知识探针：Qwen3-4B-Base 不写思维链"知道"多少？

> ℹ️ **只做分析，不能提交。** Decision 模型在 LLM 上加一个小的打分头，直接给每个选项一个概率，**不生成文字**；
> 保存出来的是自定义格式（不是 vLLM 能加载的语言模型），锁定评测没法给它打分。

结构参考 Unsloth 官方 [Qwen3.5 (4B) Decision](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(4B)-Decision.ipynb)：
`FastDecisionModel` + `DecisionTrainer`，数据是 `state`（题干）+ `questions`（4 个选项）+ `gold`（正确字母）。

**怎么用这个探针**：
1. 在公开的 MMLU-Pro STEM + SciQ 选择题上训练打分头（LoRA + 头），再在本地对 GPQA Diamond / Main **只算准确率**（以及分学科准确率、校准误差 ECE）。
2. 和 notebook 11 的生成式 CoT 分数并排：如果探针明显高于 CoT，说明知识在模型里、问题出在推理 / 格式 / 截断；如果探针也接近 25%，说明 4B 缺的是知识，SFT 数据应该往"知识"方向加（见 `experiments/DATASETS.md`）。
3. `BACKBONE` 换成 notebook 11 的 SFT 模型再跑一次：探针分数变高 = SFT 增加了知识；不变 = SFT 主要改了格式。

| 模式 | 训练 | 时间 |
|---|---|---|
| `quick` | 400 题，30 步，Diamond | T4 约 15 分钟（0.6B） |
| `a100` | 约 8000 题（MMLU-Pro STEM 6000 + SciQ 2000），1 epoch，Diamond + Main | A100 约 40 分钟（估） |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q
sys.path.insert(0, os.path.abspath("scripts"))

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth wandb {_numpy} {_pil}

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
# T4（计算能力 7.5）不支持 FlashInfer 采样器（A100 不受影响）
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "unsloth_zoo", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

In [ ]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（可选；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login, whoami
login(token = os.environ.get("HF_TOKEN"))
HF_USER = whoami(token = os.environ.get("HF_TOKEN"))["name"]
print("HF user:", HF_USER, "|", "W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

## 0. 配置

In [ ]:
EXP_NAME = "P1"
MODE     = "a100"                         # "quick" | "a100"
BACKBONE = "unsloth/Qwen3-4B-Base"        # 对比：再用 notebook 11 的 SFT 模型（PUSH_SFT_REPO）跑一次
TRAIN_MIX = {"mmlu_pro_stem": 6000, "sciq": 2000}   # a100 的数量；quick 自动缩小

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(scale=0.05, max_steps=30, epochs=1, max_seq_len=2048, lora_rank=16, lr=2e-4, batch=8, accum=2,
                 eval_configs=["gpqa_diamond"]),
    a100 = dict(scale=1.0, max_steps=-1, epochs=1, max_seq_len=2048, lora_rank=16, lr=2e-4, batch=8, accum=4,
                eval_configs=["gpqa_diamond", "gpqa_main"]),
)
Q = PRESETS[MODE]
LOAD_IN_4BIT = torch.cuda.get_device_capability()[0] < 8      # T4 用 4-bit 省显存；A100 用 16-bit
print(EXP_NAME, MODE, BACKBONE, Q, "4bit:", LOAD_IN_4BIT)

## 1. 加载（`FastDecisionModel` 给普通 LM 接一个新的打分头；头总是参与训练）

In [ ]:
import json, random
from collections import Counter, defaultdict
from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
from recipe_utils import load_mcq_items, decision_row, gpqa_choices, decontam_rows, wandb_init

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name = BACKBONE, max_seq_length = Q["max_seq_len"], load_in_4bit = LOAD_IN_4BIT)
model = FastDecisionModel.get_peft_model(
    model, r = Q["lora_rank"], lora_alpha = Q["lora_rank"], lora_dropout = 0,
    use_gradient_checkpointing = "unsloth", random_state = 3407)

RUN_NAME = f"{EXP_NAME}-decision-{BACKBONE.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, dict(
    exp = EXP_NAME, recipe = "decision_probe", mode = MODE, backbone = BACKBONE, train_mix = TRAIN_MIX,
    load_in_4bit = LOAD_IN_4BIT, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS, **Q), WANDB_ENTITY)
REPORT = dict(notebook = "14_decision_probe", exp = EXP_NAME, mode = MODE, backbone = BACKBONE,
              train_mix = TRAIN_MIX, q = Q, load_in_4bit = LOAD_IN_4BIT)

## 2. 训练数据（公开选择题 → state / questions / gold，去污染）

In [ ]:
items = []
for source, n in TRAIN_MIX.items():
    items += load_mcq_items(source, max(1, int(n * Q["scale"])), skip = 6000 if source == "sciq" else 0)
items, dropped = decontam_rows(items, lambda it: {"question": it["question"], "text": " ".join(it["choices"])},
                               "decision_train")
random.Random(0).shuffle(items)
rows = [decision_row(it["question"], it["choices"], it["answer"]) for it in items]
train_items, report = FastDecisionModel.build_dataset(rows, tokenizer, model)
print(f"skipped {report['skipped']} of {report['total']} decisions", report.get("reason") or "")
train_items, hold_items = FastDecisionModel.split_holdout(train_items, seed = 3407)
DATA = dict(train = len(train_items), holdout = len(hold_items), dropped_decontam = dropped,
            skipped = report["skipped"], by_source = Counter(it["source"] for it in items).most_common())
REPORT["data"] = DATA
print(DATA)
PROBE = {"holdout_before": FastDecisionModel.evaluate(model, tokenizer, hold_items)}   # 头是新的：约等于随机
print("holdout before training:", PROBE["holdout_before"])

## 3. 训练

In [ ]:
from transformers import TrainingArguments

trainer = DecisionTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_items,
    eval_dataset = hold_items,
    args = TrainingArguments(
        per_device_train_batch_size = Q["batch"],
        gradient_accumulation_steps = Q["accum"],
        warmup_steps = 10,
        num_train_epochs = Q["epochs"],
        max_steps = Q["max_steps"],
        learning_rate = Q["lr"],
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        bf16 = is_bfloat16_supported(),
        fp16 = not is_bfloat16_supported(),
        eval_strategy = "epoch",
        logging_steps = 10,
        output_dir = "outputs/decision",
        report_to = REPORT_TO,
        run_name = RUN_NAME,
        save_strategy = "no",
        seed = 3407,
    ),
)
stats = trainer.train()
PROBE["train_minutes"] = round(stats.metrics["train_runtime"] / 60, 1)
PROBE["calibration"] = FastDecisionModel.calibrate(model, tokenizer, hold_items)
PROBE["holdout_after"] = FastDecisionModel.evaluate(model, tokenizer, hold_items)
print(PROBE)

## 4. GPQA 探针（只算聚合数字；题目和选项不打印、不上传）

选项顺序和 notebook 11 / `recipe_utils.gpqa_choices` 一样（`Random(1000+i)`）。分学科用 GPQA 的 `High-level domain` 字段。
随机猜 = 25%。

In [ ]:
from datasets import load_dataset

def _num(d):
    return {k: round(float(v), 4) for k, v in d.items() if isinstance(v, (int, float))}

GPQA_PROBE = {}
for cfg in Q["eval_configs"]:
    ds = load_dataset("Idavidrein/gpqa", cfg, split = "train")
    rows, domains = [], []
    for i, d in enumerate(ds):
        q, choices, letter = gpqa_choices(d, i)
        rows.append(decision_row(q, choices, letter))
        domains.append(d.get("High-level domain") or "unknown")
    gp_items, rep = FastDecisionModel.build_dataset(rows, tokenizer, model)
    res = dict(n = len(gp_items), skipped = rep["skipped"], **_num(FastDecisionModel.evaluate(model, tokenizer, gp_items)))
    by_dom = defaultdict(list)
    for it in gp_items:
        by_dom[domains[it["row"]]].append(it)
    res["by_domain"] = {dom: dict(n = len(v), accuracy = round(float(FastDecisionModel.evaluate(model, tokenizer, v)["accuracy"]), 4))
                        for dom, v in sorted(by_dom.items())}
    GPQA_PROBE[cfg] = res
    print(cfg, json.dumps(res))
    if REPORT_TO == "wandb":
        wandb.run.summary[f"probe/{cfg}/accuracy"] = res.get("accuracy")
        wandb.run.summary[f"probe/{cfg}/ece"] = res.get("ece")
        for dom, v in res["by_domain"].items():
            wandb.run.summary[f"probe/{cfg}/{dom}/accuracy"] = v["accuracy"]

## 5. 汇总（最后打印的 JSON 可以贴回给 Claude；和 notebook 11 的 `acc_flex` 并排看）

In [ ]:
import os
REPORT.update(probe = {k: _num(v) if isinstance(v, dict) else v for k, v in PROBE.items()}, gpqa_probe = GPQA_PROBE,
              versions = VERSIONS, gpu = GPU_NAME, wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
for cfg, r in GPQA_PROBE.items():
    print(f"{cfg:>14s}  probe accuracy {100 * r['accuracy']:.1f}%  (random 25%, n={r['n']})")
if REPORT_TO == "wandb":
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))